In [1]:
current_step = 'step_010'

In [2]:
!apt install swig cmake ffmpeg xvfb python3-opengl
!pip install pyvirtualdisplay imageio[ffmpeg]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
Suggested packages:
  libgle3 python3-numpy python3-tk swig-doc swig-examples swig4.0-examples
  swig4.0-doc
The following NEW packages will be installed:
  freeglut3 libglu1-mesa python3-opengl swig swig4.0
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,940 kB of archives.
After this operation, 13.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 freeglut3 amd64 2.8.1-6 [74.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglu1-mesa amd64 9.0.2-1 [145 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 python3-opengl all 3.1.5+dfsg-1 [605 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/u

In [3]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

%env MUJOCO_GL=egl

from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

env: MUJOCO_GL=egl


In [4]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(log_dir, 'tensorboard_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

# best models
to_load_path = os.path.join(workDir, 'best')
print('to_load_path:', to_load_path)
algos = ["a2c", "ddpg", "ppo", "sac", "td3"]
for algo in algos:
  algo_load_path = os.path.join(to_load_path, algo)
  print('AlgoLoadPath:', algo_load_path)
  if not os.path.exists(algo_load_path):
    os.makedirs(algo_load_path)

Mounted at /content/drive
WorkDir: /content/drive/My Drive/Research/step_010
LogDir: /content/drive/My Drive/Research/step_010/20250906-151930
TfLogDir: /content/drive/My Drive/Research/step_010/20250906-151930/tensorboard_logs
to_load_path: /content/drive/My Drive/Research/step_010/best
AlgoLoadPath: /content/drive/My Drive/Research/step_010/best/a2c
AlgoLoadPath: /content/drive/My Drive/Research/step_010/best/ddpg
AlgoLoadPath: /content/drive/My Drive/Research/step_010/best/ppo
AlgoLoadPath: /content/drive/My Drive/Research/step_010/best/sac
AlgoLoadPath: /content/drive/My Drive/Research/step_010/best/td3


In [5]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

Error: /content/rl-zoo : No such file or directory
Error: /content/gym_darwin_op3 : No such file or directory
Error: /content/videos : No such file or directory


In [6]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


The directory '/content/gym_darwin_op3' does not exist - git clone
Cloning into '/content/gym_darwin_op3'...
remote: Enumerating objects: 448, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 448 (delta 88), reused 69 (delta 40), pack-reused 295 (from 1)
Receiving objects: 100% (448/448), 14.98 MiB | 30.14 MiB/s, done.
Resolving deltas: 100% (193/193), done.


In [7]:
!pip install -e {model_path}

Obtaining file:///content/gym_darwin_op3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 21.9 MB/s eta 0:00:00
  Building editable for robofei (pyproject.toml) ... done
  Created wheel for robofei: filename=robofei-0.1.22-py3-none-any.whl size=1514 sha256=895ef28450f96b6850d2be09ae0e50b39c25825c5a82f4f4063f45f962ac4bdf
  Stored in directory: /tmp/pip-ephem-wheel-cache-79p3k01d/wheels/43/c3/48/682f53e738aa575222935107724428d2ce2d742b055ebb4adc
Successfully built robofei


In [8]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


The directory '/content/rl-zoo' does not exist - git clone
Cloning into '/content/rl-zoo'...
remote: Enumerating objects: 3661, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 3661 (delta 37), reused 27 (delta 16), pack-reused 3597 (from 2)
Receiving objects: 100% (3661/3661), 8.53 MiB | 23.98 MiB/s, done.
Resolving deltas: 100% (2234/2234), done.


In [9]:
!pip install -e {trainner_path}

Obtaining file:///content/rl-zoo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 9.8 MB/s eta 0:00:00
  Building editable for rl_zoo3 (pyproject.toml) ... done
  Created wheel for rl_zoo3: filename=rl_zoo3-2.7.0-0.editable-py3-none-any.whl size=4428 sha256=095b7a66e1f87e47fed07124f35b04156f7834259a6c8ad4662d2644705ee55f
  Stored in directory: /tmp/pip-ephem-wheel-cache-wrdug51n/wheels/2d/0b/3c/be4c09b7d9d2a891b5d1bc1e086a8fe4f4b4f016a0613b625c
Successfully built rl_zoo3


In [10]:
%cd {trainner_path}

algos = {
  "a2c": {
    "lr": 7e-4, "dev": "cpu"
  },
  'ddpg': {
    'lr': 1e-3, 'dev': 'cuda'
  },
  'ppo': {
    'lr': 3e-4, 'dev': 'cpu'
  },
  'sac': {
    'lr': 3e-4, 'dev': 'cuda'
  },
  'td3': {
    'lr': 1e-3, 'dev': 'cuda'
  }
}

n_timestep = 10_000_000
save_freq = min(100_000, int(n_timestep / 10))
eval_freq = min(200_000, int(n_timestep / 10))
max_episode_steps = 1000
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]
n_envs = 16

# weights
keep_alive_reward = 1.0
ctrl_cost_weight = 0.0001
target_distance = 2.5
forward_velocity_weight = 3.0
reach_target_reward = 20.0

for algo, value in algos.items():
  print('Training:', algo)
  config = f'research_config/{algo}.yml'
  best = os.path.join(to_load_path, algo, 'best_model.zip')

  train_cmd = f'python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} \
-f "{log_dir}" --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
--vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
--env-kwargs keep_alive_reward:{keep_alive_reward} ctrl_cost_weight:{ctrl_cost_weight} \
target_distance:{target_distance} forward_velocity_weight:{forward_velocity_weight} \
reach_target_reward:{reach_target_reward} --hyperparams n_envs:{n_envs} \
learning_rate:{value["lr"]} n_timesteps:{n_timestep} env_wrapper:"{wrapper}" \
--device {value["dev"]} -i "{best}" -P '

  print(train_cmd)
  !{train_cmd}

  video_cmd = f'python3 -m rl_zoo3.record_video --algo {algo} \
--env DarwinOp3-v2 -n 3000 --load-best -o "{log_dir}" -f "{log_dir}"'
  print(video_cmd)
  !{video_cmd}


Streaming output truncated to the last 5000 lines.
| train/             |          |
|    actor_loss      | -1.87    |
|    critic_loss     | 0.00618  |
|    learning_rate   | 0.001    |
|    n_updates       | 1150337  |
---------------------------------
---------------------------------
| mean_episode/      |          |
|    control_cost    | 0.0145   |
|    forward_reward  | 2.19     |
|    health_reward   | 1        |
|    pos_x           | 0.958    |
|    pos_y           | -0.615   |
|    pos_z           | 0.28     |
|    vel_x           | 0.731    |
|    vel_y           | -0.453   |
| rollout/           |          |
|    ep_len_mean     | 316      |
|    ep_rew_mean     | 935      |
| time/              |          |
|    episodes        | 36776    |
|    fps             | 1168     |
|    time_elapsed    | 8358     |
|    total_timesteps | 9766096  |
| train/             |          |
|    actor_loss      | -1.88    |
|    critic_loss     | 0.00386  |
|    learning_rate   | 0.001   